In [2]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import MinMaxScaler
from stochastic.processes.continuous import FractionalBrownianMotion

In [8]:
SAMPLE_TO_SUBSAMPLE_SIZE = 1000
SAMPLE_SIZES = [25, 50, 100]
TRAIN_SAMPLES_NUM = 100_000
TEST_SAMPLES_NUM = 1000

In [4]:
def get_pandas_dataframe_size_in_gb(df):
    return df.memory_usage(deep=True).sum() / (1024**3)


def generate_fbm_sample(n: int, hurst: float):
    return FractionalBrownianMotion(hurst=hurst).sample(n)[
        1:
    ]  # [1:] to exclude initial 0


def gen_data(samples_num: int, sample_size: int) -> pd.DataFrame:
    """Generate fBm data with random subsampling of given size from longer trajectory."""
    data = []
    hurst_params = np.random.uniform(0.001, 0.999, size=samples_num).round(3)
    start_indices = np.random.randint(
        0, SAMPLE_TO_SUBSAMPLE_SIZE - sample_size, size=samples_num
    )

    scaler = MinMaxScaler(feature_range=(0, 1))

    for start_index, hurst_param in tqdm(
        zip(start_indices, hurst_params),
        desc=f"Generating subsamples of size {sample_size}",
    ):
        full_sample = generate_fbm_sample(SAMPLE_TO_SUBSAMPLE_SIZE, hurst_param)
        subsample = full_sample[start_index : start_index + sample_size].reshape(-1, 1)
        normalized = scaler.fit_transform(subsample).flatten()
        data.append(np.append(normalized, hurst_param))

    columns = [f"x{i}" for i in range(sample_size)] + ["hurst"]
    df = (
        pd.DataFrame(data, columns=columns)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    return df

In [6]:
os.makedirs("./data/train/", exist_ok=True)

for sample_size in SAMPLE_SIZES:
    print(f"Start generating data for sample size {sample_size} x {TRAIN_SAMPLES_NUM} samples")

    data = gen_data(samples_num=TRAIN_SAMPLES_NUM, sample_size=sample_size)

    path_to_save = f"./data/train/fbm_{sample_size}x{TRAIN_SAMPLES_NUM}.csv"

    print(f"Saving data to '{path_to_save}'")

    data.to_csv(path_to_save, index=False)

    print(f"Size: {get_pandas_dataframe_size_in_gb(data): .4f}GB")

    print("Complete.")

Start generating data for sample size 25 x 100000 samples


Generating subsamples of size 25: 100000it [00:45, 2222.09it/s]


Saving data to './data/train/fbm_25x100000.csv'
Size:  0.0194GB
Complete.
Start generating data for sample size 50 x 100000 samples


Generating subsamples of size 50: 100000it [00:47, 2091.29it/s]


Saving data to './data/train/fbm_50x100000.csv'
Size:  0.0380GB
Complete.
Start generating data for sample size 100 x 100000 samples


Generating subsamples of size 100: 100000it [00:46, 2159.82it/s]


Saving data to './data/train/fbm_100x100000.csv'
Size:  0.0753GB
Complete.


In [9]:
os.makedirs("./data/test/", exist_ok=True)

for sample_size in SAMPLE_SIZES:
    print(f"Start generating data for sample size {sample_size} x {TEST_SAMPLES_NUM} samples")

    data = gen_data(samples_num=TEST_SAMPLES_NUM, sample_size=sample_size)

    path_to_save = f"./data/test/fbm_{sample_size}x{TEST_SAMPLES_NUM}.csv"

    print(f"Saving data to '{path_to_save}'")

    data.to_csv(path_to_save, index=False)

    print(f"Size: {get_pandas_dataframe_size_in_gb(data): .4f}GB")

    print("Complete.")

Start generating data for sample size 25 x 1000 samples


Generating subsamples of size 25: 1000it [00:00, 2321.83it/s]


Saving data to './data/test/fbm_25x1000.csv'
Size:  0.0002GB
Complete.
Start generating data for sample size 50 x 1000 samples


Generating subsamples of size 50: 1000it [00:00, 2353.96it/s]


Saving data to './data/test/fbm_50x1000.csv'
Size:  0.0004GB
Complete.
Start generating data for sample size 100 x 1000 samples


Generating subsamples of size 100: 1000it [00:00, 2284.73it/s]


Saving data to './data/test/fbm_100x1000.csv'
Size:  0.0008GB
Complete.
